<a href="https://colab.research.google.com/github/sainamrathajujjavarapu9-wq/intelligent-customer-support-chatbot/blob/main/customer_support_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install -q transformers torch scikit-learn pandas numpy

In [22]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from transformers import pipeline

In [23]:
data = {
    "text": [
        "Where is my order?",
        "Can you track my order?",
        "I want to know my delivery status",
        "When will my package arrive?",
        "How can I track my shipment?",

        "I want to cancel my order",
        "Please cancel my purchase",
        "Can I cancel my order?",
        "I ordered something by mistake",
        "How do I cancel an order?",

        "I want a refund",
        "How can I get my money back?",
        "I need to request a refund",
        "Can I get a refund for my order?",
        "Please help me with a refund",

        "My product is damaged",
        "I received a damaged item",
        "The product I received is broken",
        "I got a defective product",
        "What should I do if my product is damaged?",

        "What payment methods do you accept?",
        "Can I pay using UPI?",
        "Do you accept credit cards?",
        "Can I pay with a debit card?",
        "What are the available payment options?",

        "I forgot my password",
        "How can I reset my password?",
        "I cannot login to my account",
        "Help me change my password",
        "I am unable to access my account",

        "Hello",
        "Hi",
        "Hey",
        "Good morning",
        "Good evening",

        "Thank you",
        "Thanks for your help",
        "That was helpful",
        "Thank you very much",
        "Thanks",

        "I have a problem",
        "I need help",
        "Can you help me?",
        "I have an issue",
        "I need customer support"
    ],

    "intent": [
        "order_tracking",
        "order_tracking",
        "order_tracking",
        "order_tracking",
        "order_tracking",

        "cancel_order",
        "cancel_order",
        "cancel_order",
        "cancel_order",
        "cancel_order",

        "refund",
        "refund",
        "refund",
        "refund",
        "refund",

        "damaged_product",
        "damaged_product",
        "damaged_product",
        "damaged_product",
        "damaged_product",

        "payment",
        "payment",
        "payment",
        "payment",
        "payment",

        "password",
        "password",
        "password",
        "password",
        "password",

        "greeting",
        "greeting",
        "greeting",
        "greeting",
        "greeting",

        "thanks",
        "thanks",
        "thanks",
        "thanks",
        "thanks",

        "general_help",
        "general_help",
        "general_help",
        "general_help",
        "general_help"
    ]
}

df = pd.DataFrame(data)

print("Dataset size:", len(df))
df.head()

Dataset size: 45


,text,intent
0,Where is my order?,order_tracking
1,Can you track my order?,order_tracking
2,I want to know my delivery status,order_tracking
3,When will my package arrive?,order_tracking
4,How can I track my shipment?,order_tracking


In [24]:
print(df["intent"].value_counts())

intent
order_tracking     5
cancel_order       5
refund             5
damaged_product    5
payment            5
password           5
greeting           5
thanks             5
general_help       5
Name: count, dtype: int64


In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["intent"],
    test_size=0.2,
    random_state=42,
    stratify=df["intent"]
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 36
Testing samples: 9


In [26]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

print("Transformer model loaded successfully!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Transformer model loaded successfully!


In [27]:
def get_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1)

    return embedding.numpy()[0]

In [28]:
embedding = get_embedding("Where is my order?")

print("Embedding size:", len(embedding))

Embedding size: 768


In [29]:
X_embeddings = np.array([
    get_embedding(text)
    for text in df["text"]
])

print("Embedding shape:", X_embeddings.shape)

Embedding shape: (45, 768)


In [30]:
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(df["intent"])

X_train_emb, X_test_emb, y_train_emb, y_test_emb = train_test_split(
    X_embeddings,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

classifier = LogisticRegression(max_iter=1000)

classifier.fit(X_train_emb, y_train_emb)

print("Intent classifier trained successfully!")

Intent classifier trained successfully!


In [31]:
predictions = classifier.predict(X_test_emb)

accuracy = accuracy_score(y_test_emb, predictions)

print("Chatbot Intent Classification Accuracy:",
      round(accuracy * 100, 2), "%")

Chatbot Intent Classification Accuracy: 77.78 %


In [32]:
print(
    classification_report(
        y_test_emb,
        predictions,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

                 precision    recall  f1-score   support

   cancel_order       0.00      0.00      0.00         1
damaged_product       0.50      1.00      0.67         1
   general_help       0.00      0.00      0.00         1
       greeting       1.00      1.00      1.00         1
 order_tracking       0.50      1.00      0.67         1
       password       1.00      1.00      1.00         1
        payment       1.00      1.00      1.00         1
         refund       1.00      1.00      1.00         1
         thanks       1.00      1.00      1.00         1

       accuracy                           0.78         9
      macro avg       0.67      0.78      0.70         9
   weighted avg       0.67      0.78      0.70         9



In [33]:
responses = {
    "order_tracking":
        "You can track your order using the tracking ID provided in your order confirmation.",

    "cancel_order":
        "You can request order cancellation if the order has not been shipped yet.",

    "refund":
        "You can request a refund through the orders section. Refund processing usually depends on the payment method.",

    "damaged_product":
        "I am sorry about that. Please provide your order details and contact customer support for a replacement or refund.",

    "payment":
        "We support common payment methods such as UPI, debit cards and credit cards.",

    "password":
        "You can reset your password using the 'Forgot Password' option on the login page.",

    "greeting":
        "Hello! Welcome to our customer support. How can I help you?",

    "thanks":
        "You're welcome! I'm happy to help.",

    "general_help":
        "Sure! Please tell me more about the problem you are facing."
}

In [34]:
def chatbot(user_text):

    embedding = get_embedding(user_text)

    prediction = classifier.predict([embedding])[0]

    intent = label_encoder.inverse_transform([prediction])[0]

    confidence = max(classifier.predict_proba([embedding])[0])

    if confidence < 0.40:
        return (
            "I am not completely sure I understand your question. "
            "Please contact customer support or explain your problem in more detail.",
            "unknown",
            confidence
        )

    response = responses[intent]

    return response, intent, confidence

In [35]:
questions = [
    "Where is my package?",
    "I want to cancel my order",
    "How do I get my money back?",
    "My product arrived broken",
    "Can I pay using UPI?",
    "I forgot my password",
    "Hello",
    "Thank you"
]

for question in questions:

    response, intent, confidence = chatbot(question)

    print("User:", question)
    print("Intent:", intent)
    print("Confidence:", round(confidence * 100, 2), "%")
    print("Bot:", response)
    print("-" * 60)

User: Where is my package?
Intent: order_tracking
Confidence: 80.86 %
Bot: You can track your order using the tracking ID provided in your order confirmation.
------------------------------------------------------------
User: I want to cancel my order
Intent: cancel_order
Confidence: 88.71 %
Bot: You can request order cancellation if the order has not been shipped yet.
------------------------------------------------------------
User: How do I get my money back?
Intent: refund
Confidence: 57.99 %
Bot: You can request a refund through the orders section. Refund processing usually depends on the payment method.
------------------------------------------------------------
User: My product arrived broken
Intent: damaged_product
Confidence: 72.76 %
Bot: I am sorry about that. Please provide your order details and contact customer support for a replacement or refund.
------------------------------------------------------------
User: Can I pay using UPI?
Intent: payment
Confidence: 80.38 %
Bo

In [36]:
conversation_history = []

def contextual_chatbot(user_text):

    conversation_history.append({
        "user": user_text
    })

    response, intent, confidence = chatbot(user_text)

    conversation_history[-1]["intent"] = intent
    conversation_history[-1]["bot"] = response

    return response

In [37]:
print(contextual_chatbot("Hello"))
print(contextual_chatbot("I want a refund"))
print(contextual_chatbot("Thank you"))

print("\nConversation History:")
for item in conversation_history:
    print(item)

Hello! Welcome to our customer support. How can I help you?
You can request a refund through the orders section. Refund processing usually depends on the payment method.
You're welcome! I'm happy to help.

Conversation History:
{'user': 'Hello', 'intent': 'greeting', 'bot': 'Hello! Welcome to our customer support. How can I help you?'}
{'user': 'I want a refund', 'intent': 'refund', 'bot': 'You can request a refund through the orders section. Refund processing usually depends on the payment method.'}
{'user': 'Thank you', 'intent': 'thanks', 'bot': "You're welcome! I'm happy to help."}


In [39]:
print("🤖 Intelligent Customer Support Chatbot")
print("Type 'exit' to stop.\n")

while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Bot: Thank you for contacting customer support!")
        break

    response = contextual_chatbot(user_input)

    print("Bot:", response)

🤖 Intelligent Customer Support Chatbot
Type 'exit' to stop.

You: exit
Bot: Thank you for contacting customer support!


In [41]:
import gradio as gr

def chat_function(message, history):
    response, intent, confidence = chatbot(message)

    return (
        f"{response}\n\n"
        f"🎯 **Intent:** {intent}\n"
        f"📊 **Confidence:** {confidence * 100:.1f}%"
    )


custom_css = """

/* =========================================================
   GLOBAL
========================================================= */

body {
    margin: 0 !important;
    background: #070b18 !important;
}

.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
    background:
        radial-gradient(circle at 10% 10%, rgba(99,102,241,0.20), transparent 30%),
        radial-gradient(circle at 90% 20%, rgba(168,85,247,0.18), transparent 30%),
        radial-gradient(circle at 50% 100%, rgba(14,165,233,0.12), transparent 35%),
        #070b18 !important;

    color: white !important;
    font-family: Inter, Arial, sans-serif !important;
}


/* =========================================================
   HEADER
========================================================= */

#hero {
    text-align: center;
    padding: 35px 20px 20px 20px;
}

#logo {
    width: 75px;
    height: 75px;
    margin: auto;
    border-radius: 22px;

    display: flex;
    align-items: center;
    justify-content: center;

    font-size: 38px;

    background: linear-gradient(
        135deg,
        #6366f1,
        #8b5cf6,
        #06b6d4
    );

    box-shadow:
        0 10px 35px rgba(99,102,241,0.45),
        inset 0 1px 1px rgba(255,255,255,0.3);
}

#title {
    text-align: center;
    font-size: 42px;
    font-weight: 800;

    margin-top: 18px;
    margin-bottom: 8px;

    background: linear-gradient(
        90deg,
        #ffffff,
        #c7d2fe,
        #a5b4fc
    );

    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

#subtitle {
    text-align: center;
    color: #a5b4fc;
    font-size: 17px;
    margin-bottom: 15px;
}

#status {
    display: inline-block;
    padding: 7px 15px;
    border-radius: 50px;

    background: rgba(34,197,94,0.10);
    border: 1px solid rgba(34,197,94,0.35);

    color: #86efac;
    font-size: 13px;
    font-weight: 600;
}


/* =========================================================
   CHAT AREA
========================================================= */

.chatbot {
    border-radius: 24px !important;

    border: 1px solid rgba(255,255,255,0.12) !important;

    background:
        linear-gradient(
            145deg,
            rgba(255,255,255,0.08),
            rgba(255,255,255,0.025)
        ) !important;

    backdrop-filter: blur(20px) !important;

    box-shadow:
        0 25px 70px rgba(0,0,0,0.45),
        inset 0 1px 1px rgba(255,255,255,0.08) !important;

    overflow: hidden !important;
}


/* Chat messages */

.message {
    border-radius: 18px !important;
}


/* User message */

.message.user {
    background: linear-gradient(
        135deg,
        #6366f1,
        #7c3aed
    ) !important;

    color: white !important;

    border-radius: 18px 18px 5px 18px !important;

    box-shadow:
        0 8px 25px rgba(99,102,241,0.25) !important;
}


/* Bot message */

.message.bot {
    background: rgba(255,255,255,0.07) !important;

    color: #e2e8f0 !important;

    border: 1px solid rgba(255,255,255,0.08) !important;

    border-radius: 18px 18px 18px 5px !important;
}


/* =========================================================
   INPUT
========================================================= */

textarea {
    background: rgba(255,255,255,0.06) !important;

    color: white !important;

    border: 1px solid rgba(129,140,248,0.5) !important;

    border-radius: 18px !important;

    padding: 15px !important;

    font-size: 15px !important;

    box-shadow:
        0 8px 30px rgba(0,0,0,0.25) !important;
}

textarea:focus {
    border: 1px solid #818cf8 !important;

    box-shadow:
        0 0 0 3px rgba(99,102,241,0.15),
        0 8px 30px rgba(0,0,0,0.25) !important;
}

textarea::placeholder {
    color: #94a3b8 !important;
}


/* =========================================================
   BUTTONS
========================================================= */

button {
    border-radius: 14px !important;

    font-weight: 600 !important;

    transition:
        transform 0.2s ease,
        box-shadow 0.2s ease !important;
}

button:hover {
    transform: translateY(-2px);

    box-shadow:
        0 8px 25px rgba(99,102,241,0.25) !important;
}


/* Send button */

button.primary {
    background: linear-gradient(
        135deg,
        #6366f1,
        #7c3aed
    ) !important;

    color: white !important;

    border: none !important;
}


/* =========================================================
   EXAMPLES
========================================================= */

.examples {
    margin-top: 15px !important;
}

.examples button {
    background: rgba(255,255,255,0.055) !important;

    color: #cbd5e1 !important;

    border: 1px solid rgba(255,255,255,0.10) !important;

    border-radius: 14px !important;

    padding: 10px 14px !important;
}

.examples button:hover {
    background: rgba(99,102,241,0.18) !important;

    color: white !important;

    border-color: rgba(129,140,248,0.5) !important;
}


/* =========================================================
   INFO CARDS
========================================================= */

#features {
    display: grid;

    grid-template-columns:
        repeat(3, 1fr);

    gap: 15px;

    margin-top: 25px;
}

.feature-card {
    padding: 20px;

    border-radius: 18px;

    background:
        linear-gradient(
            145deg,
            rgba(255,255,255,0.075),
            rgba(255,255,255,0.025)
        );

    border: 1px solid rgba(255,255,255,0.10);

    backdrop-filter: blur(15px);

    transition: all 0.25s ease;
}

.feature-card:hover {
    transform: translateY(-4px);

    border-color:
        rgba(129,140,248,0.45);

    box-shadow:
        0 15px 35px rgba(0,0,0,0.25);
}

.feature-icon {
    font-size: 25px;
    margin-bottom: 8px;
}

.feature-title {
    color: white;
    font-weight: 700;
    font-size: 15px;
}

.feature-text {
    color: #94a3b8;
    font-size: 13px;
    margin-top: 5px;
}


/* =========================================================
   FOOTER
========================================================= */

#footer {
    text-align: center;

    color: #64748b;

    font-size: 13px;

    margin-top: 28px;

    padding-bottom: 25px;
}

.footer-highlight {
    color: #a5b4fc;
    font-weight: 600;
}


/* =========================================================
   MOBILE
========================================================= */

@media (max-width: 768px) {

    #title {
        font-size: 30px;
    }

    #subtitle {
        font-size: 14px;
    }

    #features {
        grid-template-columns: 1fr;
    }

    #logo {
        width: 65px;
        height: 65px;
        font-size: 32px;
    }

    .chatbot {
        border-radius: 18px !important;
    }
}

"""


# =========================================================
# GRADIO UI
# =========================================================

with gr.Blocks(
    css=custom_css,
    theme=gr.themes.Base(
        primary_hue="indigo",
        neutral_hue="slate"
    )
) as demo:

    # HERO SECTION
    gr.HTML("""
        <div id="hero">

            <div id="logo">
                🤖
            </div>

            <div id="title">
                Intelligent Customer Support
            </div>

            <div id="subtitle">
                AI-powered assistance with Transformer-based NLP
            </div>

            <div id="status">
                🟢 AI Assistant Online
            </div>

        </div>
    """)


    # CHATBOT
    chatbot_ui = gr.ChatInterface(
        fn=chat_function,

        chatbot=gr.Chatbot(
            height=500,
            elem_classes="chatbot",
            show_label=False
        ),

        textbox=gr.Textbox(
            placeholder="💬 Ask me anything about your order...",
            show_label=False,
            lines=1
        ),

        examples=[
            "Where is my order?",
            "I want a refund",
            "Can I cancel my order?",
            "My product is damaged",
            "Can I pay using UPI?",
            "I forgot my password"
        ],

        title=None,
        description=None
    )


    # FEATURE CARDS
    gr.HTML("""
        <div id="features">

            <div class="feature-card">

                <div class="feature-icon">
                    🧠
                </div>

                <div class="feature-title">
                    Smart NLP
                </div>

                <div class="feature-text">
                    Understands customer questions using
                    Natural Language Processing.
                </div>

            </div>


            <div class="feature-card">

                <div class="feature-icon">
                    🎯
                </div>

                <div class="feature-title">
                    Intent Detection
                </div>

                <div class="feature-text">
                    Automatically identifies the intent
                    behind every customer message.
                </div>

            </div>


            <div class="feature-card">

                <div class="feature-icon">
                    ⚡
                </div>

                <div class="feature-title">
                    Instant Support
                </div>

                <div class="feature-text">
                    Provides fast and intelligent responses
                    to customer queries.
                </div>

            </div>

        </div>
    """)


    # FOOTER
    gr.HTML("""
        <div id="footer">

            🔐 Secure AI Customer Support

            <br><br>

            Built with
            <span class="footer-highlight">
                Python • NLP • Transformers • Gradio
            </span>

        </div>
    """)


demo.launch(share=True)

/tmp/ipykernel_2532/1957114739.py:395: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8994063a54b009a560.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
